In [2]:
# ==============================
# Step 2: Load Dataset
# ==============================
import pandas as pd
# Path relative to the notebook location (notebooks/ folder)
DATA_PATH = "../data/raw/salary.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (250000, 10)


In [9]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib

pd.set_option('display.max_columns', None)

In [3]:
# ==============================
# Step 3: Verify Dataset
# ==============================

# Preview first 5 rows
display(df.head())

# Column names
print("Columns:", df.columns.tolist())

# Data types of each column
print("\nData Types:\n", df.dtypes)

# Quick structural summary
print("\nInfo:")
df.info()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


Columns: ['job_title', 'experience_years', 'education_level', 'skills_count', 'industry', 'company_size', 'location', 'remote_work', 'certifications', 'salary']

Data Types:
 job_title             str
experience_years    int64
education_level       str
skills_count        int64
industry              str
company_size          str
location              str
remote_work           str
certifications      int64
salary              int64
dtype: object

Info:
<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   job_title         250000 non-null  str  
 1   experience_years  250000 non-null  int64
 2   education_level   250000 non-null  str  
 3   skills_count      250000 non-null  int64
 4   industry          250000 non-null  str  
 5   company_size      250000 non-null  str  
 6   location          250000 non-null  str  
 7   remote_work       250000 

In [4]:
print("Education levels:", df['education_level'].unique())
print("Company sizes:", df['company_size'].unique())
print("Remote work values:", df['remote_work'].unique())
print("Job titles (sample):", df['job_title'].unique()[:10])
print("Industries:", df['industry'].unique())

Education levels: <StringArray>
['Bachelor', 'PhD', 'High School', 'Diploma', 'Master']
Length: 5, dtype: str
Company sizes: <StringArray>
['Medium', 'Small', 'Large', 'Enterprise', 'Startup']
Length: 5, dtype: str
Remote work values: <StringArray>
['Hybrid', 'No', 'Yes']
Length: 3, dtype: str
Job titles (sample): <StringArray>
[              'AI Engineer',              'Data Analyst',
        'Frontend Developer',          'Business Analyst',
           'Product Manager',         'Backend Developer',
 'Machine Learning Engineer',           'DevOps Engineer',
         'Software Engineer',     'Cybersecurity Analyst']
Length: 10, dtype: str
Industries: <StringArray>
[   'Healthcare',       'Telecom',         'Media',        'Retail',
 'Manufacturing',     'Education',       'Finance',    'Technology',
    'Consulting',    'Government']
Length: 10, dtype: str


In [5]:
# ==============================
# Step 4: Separate Features (X) and Target (y)
# ==============================

# Target variable — what we want to predict
y = df["salary"]

# Features — everything except the target
X = df.drop(columns=["salary"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:", X.columns.tolist())

X shape: (250000, 9)
y shape: (250000,)

Feature columns: ['job_title', 'experience_years', 'education_level', 'skills_count', 'industry', 'company_size', 'location', 'remote_work', 'certifications']


In [6]:
print(df['location'].nunique())

10


In [7]:
# ==============================
# Step 5: Identify Numerical and Categorical Features
# ==============================

numerical_features = ["experience_years", "skills_count", "certifications"]

ordinal_features = ["education_level", "company_size", "remote_work"]

nominal_features = ["job_title", "industry", "location"]

# Sanity check: do these three lists cover every column in X exactly once?
all_features = numerical_features + ordinal_features + nominal_features
print("Total listed:", len(all_features))
print("Total in X:", X.shape[1])
print("Match:", set(all_features) == set(X.columns))

Total listed: 9
Total in X: 9
Match: True


In [10]:
# ==============================
# Step 6: Encode Categorical Features
# ==============================

# --- Define explicit category orders for Ordinal columns ---
education_order = ["High School", "Diploma", "Bachelor", "Master", "PhD"]
company_size_order = ["Startup", "Small", "Medium", "Large", "Enterprise"]
remote_work_order = ["No", "Hybrid", "Yes"]

# --- Ordinal Encoder (needs explicit order per column) ---
ordinal_encoder = OrdinalEncoder(
    categories=[education_order, company_size_order, remote_work_order]
)

# --- One-Hot Encoder (no order needed) ---
onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# --- Quick test: fit and transform ordinal features ---
ordinal_encoded = ordinal_encoder.fit_transform(X[ordinal_features])
print("Ordinal encoded shape:", ordinal_encoded.shape)
print("Sample:\n", ordinal_encoded[:5])

# --- Quick test: fit and transform nominal features ---
nominal_encoded = onehot_encoder.fit_transform(X[nominal_features])
print("\nOne-hot encoded shape:", nominal_encoded.shape)

Ordinal encoded shape: (250000, 3)
Sample:
 [[2. 2. 1.]
 [2. 1. 0.]
 [4. 2. 0.]
 [4. 2. 2.]
 [2. 3. 2.]]

One-hot encoded shape: (250000, 32)


In [11]:
# ==============================
# Step 7: Scale Numerical Features
# ==============================

scaler = StandardScaler()

# Quick test: fit and transform numerical features
numerical_scaled = scaler.fit_transform(X[numerical_features])

print("Numerical scaled shape:", numerical_scaled.shape)
print("\nSample (scaled):\n", numerical_scaled[:5])

print("\nMean (should be ~0):", numerical_scaled.mean(axis=0))
print("Std (should be ~1):", numerical_scaled.std(axis=0))

Numerical scaled shape: (250000, 3)

Sample (scaled):
 [[-8.92322321e-04 -1.45964725e+00 -2.88271936e-01]
 [-8.25894468e-01  1.27794008e+00 -1.46028059e+00]
 [ 1.31911111e+00 -1.09463561e+00 -8.74276261e-01]
 [ 1.48411154e+00  5.47916789e-01 -1.46028059e+00]
 [ 8.24109823e-01 -5.47118144e-01 -1.46028059e+00]]

Mean (should be ~0): [ 1.20280674e-16  7.04858394e-17 -5.91171556e-17]
Std (should be ~1): [1. 1. 1.]


In [12]:
# ==============================
# Step 8: Build a Preprocessing Pipeline
# ==============================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("ord", OrdinalEncoder(
            categories=[education_order, company_size_order, remote_work_order]
        ), ordinal_features),
        ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal_features)
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['experience_years', 'skills_count',
                                  'certifications']),
                                ('ord',
                                 OrdinalEncoder(categories=[['High School',
                                                             'Diploma',
                                                             'Bachelor',
                                                             'Master', 'PhD'],
                                                            ['Startup', 'Small',
                                                             'Medium', 'Large',
                                                             'Enterprise'],
                                                            ['No', 'Hybrid',
                                                             'Yes']]),
                                 ['education_level', 'company_size',
                        

In [13]:
# ==============================
# Step 9: Train-Test Split
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (200000, 9)
X_test shape: (50000, 9)
y_train shape: (200000,)
y_test shape: (50000,)


In [14]:
# ==============================
# Step 10: Save the Preprocessor
# ==============================

# Fit the preprocessor ONLY on training data
preprocessor.fit(X_train)

# Transform both train and test sets using what was learned from training
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

# Save the FITTED preprocessor to disk
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(preprocessor, "../models/preprocessor.pkl")

print("\nPreprocessor saved successfully to ../models/preprocessor.pkl")

X_train_processed shape: (200000, 38)
X_test_processed shape: (50000, 38)

Preprocessor saved successfully to ../models/preprocessor.pkl
